In [ ]:
%run ./NB_Config_SELLING_SYSTEM

from datetime import datetime
import re
import uuid
from pyspark.sql import functions as F

PIPELINE_RUN_ID = str(uuid.uuid4())
start_time = datetime.utcnow()
results = []

def _schema_key(column_name):
    return re.sub(r'[^a-z0-9]', '', str(column_name).lower())

def active_bronze(table_name):
    return (spark.read.format('delta').load(f'{BRONZE_LH_ABFSS}/{BRONZE_SCHEMA}/{table_name}')
                .filter(F.col('_IS_DELETED') == False))

def project_final_schema(df, final_columns):
    # Resolve source columns by exact name, then case/format-insensitive name.
    available = {_schema_key(column_name): column_name for column_name in df.columns}
    expressions = []
    missing = []
    for final_column in final_columns:
        source_column = final_column if final_column in df.columns else available.get(_schema_key(final_column))
        if source_column is None:
            expressions.append(F.lit(None).cast('string').alias(final_column))
            missing.append(final_column)
        else:
            expressions.append(F.col(source_column).alias(final_column))
    return df.select(*expressions), missing

for table_name, final_columns in FINAL_COLUMNS.items():
    source_name = FINAL_PRIMARY_SOURCE[table_name]
    df = active_bronze(source_name)
    df, missing = project_final_schema(df, final_columns)
    df = (df
            .withColumn('_PIPELINE_NAME', F.lit(PIPELINE_NAME))
            .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
            .withColumn('_UPDATED_AT', F.current_timestamp()))
    output_path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{table_name}'
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(output_path)
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.{table_name} "
        f"USING DELTA LOCATION '{output_path}'"
    )
    row_count = df.count()
    results.append({'table': table_name, 'source': source_name, 'rows': row_count, 'missing_columns': missing})
    if missing:
        print(f'{table_name}: {source_name} -> {len(final_columns)} columns; unresolved: {missing}')
    else:
        print(f'{table_name}: {source_name} -> {len(final_columns)} renamed/projected columns')

print(f'Silver complete: {len(results)} final schemas written and registered')

In [ ]:
# Review transformation logic that requires joins/formulas beyond direct projection.
for table_name, logic in FINAL_LOGIC.items():
    derived = {column: expression for column, expression in logic.items() if not str(expression).startswith('Direct extract')}
    if derived:
        print(f'\n{table_name}: {len(derived)} derived or joined columns')
        for column, expression in list(derived.items())[:10]:
            print(f'  {column}: {expression}')